In [2]:
df_products = spark.table("lh_retail_silver.dbo.products")
df_reviews = spark.table("lh_retail_silver.dbo.product_reviews")

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 4, Finished, Available, Finished, False)

In [ ]:
# Management wants to know which categories and brands perform best based on price, stock, rating, and discount levels
from pyspark.sql import functions as F

df_brand_category_performance = (df_products.groupBy("category", "brand")
                                    .agg(
                                        F.round(F.avg("price"),2).alias("avg_price"),
                                          F.sum("stock").alias("total_stock"),
                                          F.round( F.avg("rating"),2).alias("avg_rating"),
                                          F.round(F.avg("discountPercentage"),2).alias("avg_discount"))
                                           
                                )
                                            
display(df_brand_category_performance)

In [ ]:
from pyspark.sql.window import Window

window_spec = (
    Window
    .partitionBy("category")
    .orderBy(F.col("avg_rating").desc())
)

df_brand_category_ranked = (
    df_brand_category_performance
    .withColumn(
        "rating_rank",
        F.dense_rank().over(window_spec)
    )
).orderBy("category","rating_rank")

display(df_brand_category_ranked)

In [ ]:
# Operations wants to identify products with low stock and possible replenishment risk

df_inventory_risk = (
    df_products
    .withColumn(
        "inventory_risk",
        F.when(F.col("stock") <= 10, "Critical")
         .when(F.col("stock") <= 30, "Low")
         .when(F.col("stock") <= 60, "Medium")
         .otherwise("Healthy")
    )
)

display(df_inventory_risk)


In [ ]:
# Operations wants to identify products with low stock and possible replenishment risk final

df_inventory_risk = (
    df_inventory_risk
    .select(
        "id",
        "title",
        "category",
        "brand",
        "stock",
        "availabilityStatus",
        "minimumOrderQuantity",
        "inventory_risk"
    )
)

display(
    df_inventory_risk.orderBy(F.col("stock").asc())
)

In [ ]:
# Customer experience wants to know which products receive the best and worst reviews

df_reviews_categorized = (
    df_reviews
    .withColumn(
        "review_category",
        F.when(F.col("review_rating") >= 4, "Good")
         .when(F.col("review_rating") == 3, "Fair")
         .otherwise("Bad")
    )
)

display(df_reviews_categorized)

In [ ]:
df_review_summary = (
    df_reviews_categorized
    .groupBy("product_id")
    .agg(
        F.round(F.avg("review_rating"), 2).alias("avg_review_rating"),
        F.count("*").alias("total_reviews"),

        F.sum(
            F.when(F.col("review_category") == "Good", 1).otherwise(0)
        ).alias("good_reviews"),

        F.sum(
            F.when(F.col("review_category") == "Fair", 1).otherwise(0)
        ).alias("fair_reviews"),

        F.sum(
            F.when(F.col("review_category") == "Bad", 1).otherwise(0)
        ).alias("bad_reviews")
    )
)
display(df_review_summary)

In [ ]:
df_reviews_summary_final= df_review_summary.join(
    df_products,
    df_review_summary.product_id == df_products.id,
    "inner"
)
display(df_reviews_summary_final)

In [ ]:
# Customer experience wants to know which products receive the best and worst reviews final with business relevant columns

df_customer_review_performance = (
    df_reviews_summary_final
    .select(
        "product_id",
        "title",
        "category",
        "brand",
        "avg_review_rating",
        "total_reviews",
        "good_reviews",
        "fair_reviews",
        "bad_reviews"
    )
    .orderBy(F.col("avg_review_rating").desc())
)

display(df_customer_review_performance)

In [ ]:
# Review status/classification at the product level

df_customer_review_performance = (
    df_customer_review_performance
    .withColumn(
        "review_performance",
        F.when(F.col("avg_review_rating") >= 4, "Strong")
         .when(F.col("avg_review_rating") >= 3, "Average")
         .otherwise("Weak")
    )
)

display(df_customer_review_performance)

In [ ]:
# Commercial team wants category-level KPIs such as average price, average rating, total stock, and average discount

df_category_performance_kpi = (df_products.groupBy("category")
                                    .agg(
                                        F.round(F.avg("price"),2).alias("avg_price"),
                                          F.sum("stock").alias("total_stock"),
                                          F.count("*").alias("product_count"),
                                          F.round( F.avg("rating"),2).alias("avg_rating"),
                                          F.round(F.avg("discountPercentage"),2).alias("avg_discount"))
                                           
                                )
                                            
display(df_category_performance_kpi)

In [ ]:
dim_product = (
    df_products
    .select(
        F.col("id").alias("product_id"),
        "sku",
        "title",
        "category",
        "brand",
        F.col("availabilityStatus").alias("availability_status")
    )
    .withColumn(
        "product_key",
        F.xxhash64("product_id", "sku")
    )
)
display(dim_product)

In [ ]:
# reorder the dim table columns properly for analytics

dim_product = dim_product.select(
    "product_key",
    "product_id",
    "sku",
    "title",
    "category",
    "brand",
    "availability_status"
)

display(dim_product)

In [54]:
(
    df_brand_category_ranked
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_gold.dbo.brand_category_performance")
)

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 56, Finished, Available, Finished, False)

In [55]:
(
    df_inventory_risk
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_gold.dbo.inventory_risk")
)

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 57, Finished, Available, Finished, False)

In [56]:
(
    df_customer_review_performance
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_gold.dbo.customer_review_performance")
)

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 58, Finished, Available, Finished, False)

In [57]:
(
    df_category_performance_kpi
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_gold.dbo.category_performance_kpi")
)

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 59, Finished, Available, Finished, False)

In [58]:
(
    dim_product
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("lh_retail_gold.dbo.dim_product")
)

StatementMeta(, 69d665ce-e029-46ae-9d08-294af097e3ea, 60, Finished, Available, Finished, False)